In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import sys, platform, json, importlib.metadata, torch
assert not torch.cuda.is_available(), 'CPU teaching lane unexpectedly sees CUDA'
identity = dict(executable=sys.executable, prefix=sys.prefix, python=platform.python_version(), platform=platform.platform(), cuda_available=torch.cuda.is_available(), torch_cuda_build=torch.version.cuda, packages={name:importlib.metadata.version(name) for name in ['torch','numpy','matplotlib','nbformat','nbclient','ipykernel','transformers','tokenizers','peft']})
print('__DONGXI_KERNEL_IDENTITY__:' + json.dumps(identity, sort_keys=True))

# A checkpoint includes the meanings of its IDs

Chapter 1 · Day 1 · CPU, local fixtures only

A model can accept an eight-row embedding table while a different tokenizer changes what row 2 means. Should successful loading establish compatibility? Predict the failure before checking. We will intervene on vocabulary, encoding rules and the generation interface, then distinguish them from per-invocation metadata. Nothing here downloads or evaluates a pretrained model.

In [ ]:
from pathlib import Path
import sys, json, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms').exists())
sys.path.insert(0, str(ROOT / 'src'))
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers
from transformers import PreTrainedTokenizerFast, AutoTokenizer
import matplotlib.pyplot as plt
from dongxi_llms.run_identity import tokenizer_interface, assert_compatible, IdentityJournal
REVISION = '1' * 40  # An authored fixture declaration, not an upstream commit.
TEMPLATE = '{% for message in messages %}{{ message.content }} {% endfor %}'
def fixture(permuted=False, lowercase=False):
    vocab = {'[UNK]': 0, '[EOS]': 1, 'cat': 2, 'dog': 3, 'red': 4, 'blue': 5, 'hello': 6, '[BOS]': 7}
    if permuted:
        vocab['cat'], vocab['dog'] = vocab['dog'], vocab['cat']
    backend = Tokenizer(models.WordLevel(vocab, unk_token='[UNK]'))
    backend.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
    if lowercase:
        backend.normalizer = normalizers.Lowercase()
    tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend, unk_token='[UNK]',
        eos_token='[EOS]', pad_token='[EOS]', bos_token='[BOS]')
    tokenizer.chat_template = TEMPLATE
    return tokenizer
def inspect(tokenizer, **changes):
    config = dict(template=TEMPLATE, stop_ids=[1], tokenizer_id='original-local-fixture',
                  tokenizer_revision=REVISION)
    config.update(changes)
    return tokenizer_interface(tokenizer, **config)
original, permuted, normalized = fixture(), fixture(permuted=True), fixture(lowercase=True)

Prediction → implementation: both tokenizers have eight IDs. Is that sufficient? Use the adjacent reference to compare the *same* text and the inverse mapping. The model is not given the spelling of an ID; an unchanged tensor shape cannot repair a changed label.

In [ ]:
baseline = inspect(original)
changed_mapping = inspect(permuted)
assert len(original) == len(permuted) == 8
print('Original cat dog:', original.encode('cat dog'))
print('Permuted cat dog:', permuted.encode('cat dog'))
try:
    assert_compatible(baseline, changed_mapping)
except ValueError as error:
    print('Rejected as intended:', error)
else:
    raise AssertionError('Unsafe mapping was accepted')
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.axis('off')
inverse = lambda t: {v: k for k, v in t.get_vocab().items()}
rows = [[inverse(t)[i] for i in range(8)] for t in (original, permuted)]
table = ax.table(cellText=rows, rowLabels=['Parent tokenizer', 'Permuted tokenizer'],
    colLabels=[f'ID {i}' for i in range(8)], cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1, 1.8)
for row in (1, 2):
    for column in (2, 3):
        table[(row, column)].set_facecolor('#f7ddc7')
ax.set_title('Measured fixture mappings: equal table size, different ID meanings', pad=10)
plt.show()

**Saved reference preview — rerun the preceding cell for current mappings.**

![Equal-sized tokenizer mappings](../figures/chapter-01/day-01-04_checkpoint_interface-01.png)

This fixed image records the authored CPU fixture. Changing the fixture does not change the saved preview.

Perturbation: keep the vocabulary unchanged but lowercase incoming text. Then change only stop IDs, the template, or the declared revision. Which checks should fail? Reference: mapping equality is necessary but not sufficient. An encoding rule can map `CAT` to an unknown ID in one tokenizer and `cat` in the other. A new template or stop policy changes the interface even when the vocabulary stays fixed.

In [ ]:
assert original.get_vocab() == normalized.get_vocab()
print('Original CAT:', original.encode('CAT'))
print('Lowercasing CAT:', normalized.encode('CAT'))
changes = {
    'encoding': inspect(normalized),
    'stop ID': inspect(original, stop_ids=[7]),
    'template': inspect(original, template=TEMPLATE + 'different ending'),
    'revision declaration': inspect(original, tokenizer_revision='2' * 40),
}
for name, changed in changes.items():
    try:
        assert_compatible(baseline, changed)
    except ValueError as error:
        print(name, '→ rejected:', error)
    else:
        raise AssertionError(f'{name} was silently accepted')
# A declaration is separate evidence: matching semantics do not authenticate ancestry.
assert assert_compatible(baseline, changes['revision declaration'], compare_source=False)

Integration: an ordinary local Hugging Face save/reload should *not* be treated as an incompatible tokenizer. Run the reference, then enable runtime padding/truncation and explain why those temporary backend buffers are excluded from the semantic hash. The chosen padding/truncation policy still belongs in the stable training configuration.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    original.save_pretrained(directory)
    restored = AutoTokenizer.from_pretrained(directory, local_files_only=True)
    assert assert_compatible(baseline, inspect(restored))
    restored.backend_tokenizer.enable_padding(pad_id=1, pad_token='[EOS]')
    restored.backend_tokenizer.enable_truncation(5)
    assert assert_compatible(baseline, inspect(restored))
print('Actual local save/reload and transient-buffer checks passed.')

Failure evidence: a run can fail before loading the model. What identity should survive? Reference: input/source/lock evidence already captured plus the last stage. This controlled failure tests a journal, not real hardware. A per-invocation ID is useful evidence, but putting it or a timestamp into checkpoint equality would reject every legitimate restart.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    journal = IdentityJournal(directory, {'seed': 61, 'token': 'do-not-record'}, invocation_id='authored-fixture')
    journal.attach({'checkpoint_interface': baseline, 'boundary': 'controlled journal test only'})
    journal.stage('checking_parent_interface')
    journal.fail(ValueError('authored mismatch control'))
    saved = json.loads(journal.path.read_text())
    assert saved['failure']['stage'] == 'checking_parent_interface'
    assert saved['identity']['checkpoint_interface'] == baseline
    assert saved['config']['token'] == '[REDACTED]'
    print('Failure evidence retained:', saved['status'], saved['failure']['stage'])

Interpretation checkpoint: does a matching fingerprint prove better language, verified upstream ancestry, or a working GB10 kernel? No. It rules out the tested interface changes under this fast-tokenizer serialization contract. Slow/custom tokenizers fail closed until their encoding contract is explicitly supported. See Chapter 1 §1.10 and its worked solutions for the handoff argument. The independent regression suite also exercises a tiny random HF model and a nonzero PEFT merge/reload; neither is a Qwen capability experiment.